# EPCLM-X4C-001 — X4 Cellular Computation Redesign

Tesla T4、Internet、已有 GITHUB_TOKEN Secret；Save & Run All。
自动执行 G0 正确性与固定数据身份 → G1 1M 开发筛选（100K 首个里程碑）→ 仅在实测阈值通过后进入 G2，再按机制证据决定 G3。
每个语言模型独立初始化、原 Native 125 Token / Batch 8 / FP16+GradScaler；推理统一 FP32。原 Phase-C 数据 Manifest 与 Tokenizer 按 Hash 核验。
最多两小时：5400 秒执行、600 秒发布、1200 秒余量；时间不足保存诚实的 INCOMPLETE。10M 和三 Seed 复核不自动执行。
最终权重与原始训练 Trace 仅保存在本次 Kaggle Output，不进入 Git。单阶段完成不代表全部假设成立。


In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

BRANCH = 'codex/epclm-x4c-001'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-x4c-001'
OUTPUT = WORKSPACE / 'epclm-x4c-001-bootstrap'
HARD_LIMIT_SECONDS = 7200
deadline = time.monotonic() + HARD_LIMIT_SECONDS

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix': os.killpg(process.pid, signal.SIGTERM)
        else: process.terminate()
        try: output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix': os.killpg(process.pid, signal.SIGKILL)
            else: process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Notebook time limit reached; partial reports and final models, if any, remains under ' + str(OUTPUT))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Output is under ' + str(OUTPUT))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'switch', '-C', BRANCH, 'FETCH_HEAD'])
    run_bounded(['git', '-C', str(REPO), 'clean', '-fd'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH,
        REPOSITORY, str(REPO)])

SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT = WORKSPACE / ('epclm-x4c-001-' + SOURCE_COMMIT[:12])

import importlib.util
missing = [name for name in ('datasets','tokenizers','huggingface_hub','pytest') if importlib.util.find_spec(name) is None]
if missing:
    run_bounded([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing, timeout_seconds=600)
execution_seconds = min(5400, deadline - time.monotonic() - 1860)
if execution_seconds < 60:
    raise TimeoutError('Insufficient two-hour budget after setup; training was not started.')
run_bounded([sys.executable, str(REPO / 'research/projects/cellular-language-models/experiments/epclm-x4c-001/train.py'),
    '--stage', 'auto', '--profile', 'dev', '--match', 'tokens', '--output', str(OUTPUT),
    '--execution-seconds', str(execution_seconds)], cwd=REPO, timeout_seconds=6000)
